# RSNA — DenseNet-121 y validación externa cruzada entre hospitales

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental,
> no validado clínicamente.

Entrena en **RSNA** (urgencias de EE. UU., 26.684 radiografías con opacidad
pulmonar anotada por radiólogos) y evalúa fuera de distribución en:

1. **NIH ChestX-ray14** — otro sistema hospitalario, etiqueta de neumonía
   extraída por NLP. Es el diseño de Zech et al. (PLOS Medicine 2018), que
   encontró caídas significativas al cruzar de hospital.
2. **Chest X-Ray Images (Pneumonia)** — población pediátrica de Guangzhou.

Criterio del proyecto: una caída de AUROC > 0,10 se documenta y se analiza.

Código: https://github.com/GGGuardin/chest-xray-pneumonia

In [ ]:
import subprocess, sys, os, time, glob
T0 = time.time()

subprocess.run(['rm', '-rf', '/tmp/repo'], check=False)
subprocess.run(['git', 'clone', '--depth', '1', '-q',
                'https://github.com/GGGuardin/chest-xray-pneumonia.git', '/tmp/repo'], check=True)
os.chdir('/tmp/repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations', 'pydicom'], check=True)

import torch
print('torch', torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit('Sin GPU disponible')
nombre = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
soportadas = torch.cuda.get_arch_list()
print(f'GPU: {nombre} | capacidad sm_{cap[0]}{cap[1]} | binario soporta {soportadas}')
assert f'sm_{cap[0]}{cap[1]}' in soportadas, (
    f'PyTorch no soporta esta GPU ({nombre}). Relanza el kernel con T4.')
torch.zeros(8, device='cuda').sum().item()
print('Comprobacion CUDA: OK')

## 1. Localización de los datos montados

Las rutas se descubren buscando ficheros ancla: Kaggle ha cambiado el punto de
montaje entre versiones y codificarlas a mano es frágil.

In [ ]:
OUT = '/kaggle/working'

anclas = glob.glob('/kaggle/input/**/stage_2_train_labels.csv', recursive=True)
if not anclas:
    print('Arbol de /kaggle/input:')
    for r, d, f in os.walk('/kaggle/input'):
        if r.count(os.sep) <= 5:
            print(' ', r, '->', (d[:4] or f[:4]))
    raise SystemExit('No encuentro stage_2_train_labels.csv')
RSNA = os.path.dirname(anclas[0])

nih_csv = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
NIH = os.path.dirname(nih_csv[0]) if nih_csv else None
ped = [p for p in glob.glob('/kaggle/input/**/chest_xray', recursive=True) if os.path.isdir(p)]
PED = ped[0] if ped else None

print('RSNA      :', RSNA)
print('NIH       :', NIH)
print('PEDIATRICO:', PED)
print('DICOM detectados:', len(glob.glob(os.path.join(RSNA, 'stage_2_train_images', '*.dcm'))))

## 2. Manifiesto y split POR PACIENTE

Lee sexo, edad y proyección AP/PA de las cabeceras DICOM (unos minutos sobre
26k ficheros). Sin esos metadatos no hay análisis de subgrupos.

In [ ]:
!python -m src.prepare_data --dataset rsna --root {RSNA} --out {OUT}/manifest_rsna.csv

In [ ]:
import pandas as pd
from src.data import split_summary

df = pd.read_csv(f'{OUT}/manifest_rsna.csv')
print(split_summary(df).to_string())
print('\nProyeccion:\n', df['view'].value_counts().to_string())
print('\nSexo:\n', df['sex'].value_counts().to_string())
print('\nEdad: mediana %.0f' % df.age.median())
assert (df.groupby('patient_id')['split'].nunique() == 1).all(), 'FUGA DE DATOS entre splits'
print('\nOK: ningun paciente aparece en mas de un split.')

## 3. Entrenamiento

In [ ]:
!python -m src.train --config configs/rsna.yaml \
    --manifest {OUT}/manifest_rsna.csv \
    --out-dir {OUT}/runs/rsna_densenet121 \
    --batch-size 64

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs(f'{OUT}/reports', exist_ok=True)
h = pd.read_csv(f'{OUT}/runs/rsna_densenet121/history.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label='train'); ax[0].plot(h.epoch, h.val_loss, label='val')
ax[0].set_title('loss'); ax[0].set_xlabel('epoca'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h.epoch, h.train_auroc, label='train'); ax[1].plot(h.epoch, h.val_auroc, label='val')
ax[1].axhline(0.85, ls='--', color='grey', label='umbral del proyecto 0,85')
ax[1].set_title('AUROC'); ax[1].set_xlabel('epoca'); ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(f'{OUT}/reports/curvas_entrenamiento.png', dpi=150)
print(h.to_string(index=False))

## 4. Evaluación en test interno (split por paciente)

In [ ]:
!python -m src.evaluate --checkpoint {OUT}/runs/rsna_densenet121/best.pth \
    --manifest {OUT}/manifest_rsna.csv --split test --out-dir {OUT}/reports/rsna_test

## 5. Grad-CAM y auditoría de shortcut learning

In [ ]:
!python -m src.explain --checkpoint {OUT}/runs/rsna_densenet121/best.pth \
    --manifest {OUT}/manifest_rsna.csv --split test --n 16 --out-dir {OUT}/reports/gradcam

## 6. Infradiagnóstico por subgrupos

In [ ]:
!python -m src.fairness --predictions {OUT}/reports/rsna_test/predictions.csv \
    --out-dir {OUT}/reports/fairness

## 7. Validación externa cruzada

**NIH** cambia de hospital manteniendo población adulta — es el contraste de
Zech et al. Ojo a la asimetría de etiqueta: RSNA marca *opacidad pulmonar*
anotada por radiólogos y NIH marca *neumonía* extraída por NLP. No son la misma
definición, así que parte de la caída será desplazamiento de etiqueta y no solo
de dominio; eso hay que decirlo, no esconderlo.

**Pediátrico** cambia además de edad, país y prevalencia (74%).

In [ ]:
CKPT = f'{OUT}/runs/rsna_densenet121/best.pth'

if NIH:
    !python -m src.prepare_data --dataset nih --root {NIH} --target Pneumonia \
        --out {OUT}/manifest_nih.csv
    !python -m src.evaluate --checkpoint {CKPT} --manifest {OUT}/manifest_nih.csv \
        --split all --out-dir {OUT}/reports/externo_nih
else:
    print('NIH no montado.')

if PED:
    !python -m src.prepare_data --dataset kaggle_pneumonia --root {PED} \
        --out {OUT}/manifest_ped.csv
    !python -m src.evaluate --checkpoint {CKPT} --manifest {OUT}/manifest_ped.csv \
        --split all --out-dir {OUT}/reports/externo_pediatrico
else:
    print('Pediatrico no montado.')

## 8. Resumen

In [ ]:
import json, shutil

resumen = {'entrenado_en': 'rsna', 'umbral_proyecto_auroc': 0.85}
for nombre, ruta in [('test_interno_rsna', f'{OUT}/reports/rsna_test/metrics.json'),
                     ('externo_nih', f'{OUT}/reports/externo_nih/metrics.json'),
                     ('externo_pediatrico', f'{OUT}/reports/externo_pediatrico/metrics.json')]:
    if os.path.exists(ruta):
        with open(ruta) as f:
            resumen[nombre] = json.load(f)

for ruta in [f'{OUT}/reports/fairness/fairness.json', f'{OUT}/reports/gradcam/shortcut_audit.json']:
    if os.path.exists(ruta):
        with open(ruta) as f:
            d = json.load(f)
        resumen[os.path.basename(os.path.dirname(ruta))] = {k: v for k, v in d.items() if k != 'detalle'}

interno = resumen.get('test_interno_rsna', {}).get('auroc')
if interno:
    for clave in ('externo_nih', 'externo_pediatrico'):
        ext = resumen.get(clave, {}).get('auroc')
        if ext:
            resumen[f'caida_auroc_{clave}'] = round(interno - ext, 4)
resumen['minutos_totales'] = round((time.time() - T0) / 60, 1)

with open(f'{OUT}/resumen.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False))

for f_ in glob.glob(f'{OUT}/manifest_*.csv'):
    shutil.move(f_, '/tmp/' + os.path.basename(f_))

print('\nArchivos de salida:')
for p in sorted(glob.glob(f'{OUT}/**/*', recursive=True)):
    if os.path.isfile(p):
        print(f'  {os.path.getsize(p)/1e6:8.2f} MB  {p}')